# Coconut Leaf Die Back Detection Model (v3)

## Supervisor Requirements:
- ✅ No Data Leaking (separate train/val/test, augmentation only on train)
- ✅ No Overfitting (Dropout, L2 Regularization, Early Stopping)
- ✅ Class-wise Precision, Recall, F1-Score
- ✅ P, R, F1 should be close to each other
- ✅ Similar values across all classes
- ✅ Accuracy close to F1

## Classes:
1. healthy
2. leaf_die_back
3. not_cocount

## 1. Import Libraries

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# TensorFlow
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# Sklearn
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support, accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 2. Configuration

In [ ]:
# Model Configuration
MODEL_VERSION = "v3"
MODEL_NAME = "leaf_dieback"

# Paths - UPDATED to leaf_dieback_v1
BASE_DIR = r"C:\Users\Tharindu Nandun\Desktop\Research\Research\ml"
DATA_DIR = os.path.join(BASE_DIR, "data", "processed", "leaf_dieback_v1")
MODEL_DIR = os.path.join(BASE_DIR, "models", f"{MODEL_NAME}_{MODEL_VERSION}")

TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")
TEST_DIR = os.path.join(DATA_DIR, "test")

# Create model directory
os.makedirs(MODEL_DIR, exist_ok=True)

# Training Configuration - Anti-Overfitting
CONFIG = {
    'img_size': 224,
    'batch_size': 32,
    'epochs': 50,
    'learning_rate': 0.0001,
    'dropout_rate': 0.5,          # High dropout to prevent overfitting
    'l2_reg': 0.01,               # L2 regularization
    'patience_early_stop': 10,    # Early stopping patience
    'patience_reduce_lr': 5       # LR reduction patience
}

print("Configuration:")
print(f"  Model: {MODEL_NAME}_{MODEL_VERSION}")
print(f"  Data: {DATA_DIR}")
print(f"  Model save: {MODEL_DIR}")
print(f"\nAnti-Overfitting Settings:")
print(f"  Dropout: {CONFIG['dropout_rate']}")
print(f"  L2 Reg: {CONFIG['l2_reg']}")
print(f"  Early Stop: {CONFIG['patience_early_stop']} epochs")

## 3. Dataset Analysis

In [ ]:
# Count images per class
def count_images(directory):
    counts = {}
    for class_name in sorted(os.listdir(directory)):
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            count = len([f for f in os.listdir(class_path) 
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            counts[class_name] = count
    return counts

train_counts = count_images(TRAIN_DIR)
val_counts = count_images(VAL_DIR)
test_counts = count_images(TEST_DIR)

print("="*60)
print("DATASET SUMMARY")
print("="*60)
print(f"{'Class':<15} {'Train':>8} {'Val':>8} {'Test':>8} {'Total':>8}")
print("-"*60)
for class_name in train_counts.keys():
    total = train_counts[class_name] + val_counts.get(class_name, 0) + test_counts.get(class_name, 0)
    print(f"{class_name:<15} {train_counts[class_name]:>8} {val_counts.get(class_name, 0):>8} {test_counts.get(class_name, 0):>8} {total:>8}")
print("-"*60)
print(f"{'TOTAL':<15} {sum(train_counts.values()):>8} {sum(val_counts.values()):>8} {sum(test_counts.values()):>8} {sum(train_counts.values())+sum(val_counts.values())+sum(test_counts.values()):>8}")

NUM_CLASSES = len(train_counts)
print(f"\nNumber of classes: {NUM_CLASSES}")

In [ ]:
# Dataset Distribution Chart
fig, ax = plt.subplots(figsize=(12, 6))

classes = list(train_counts.keys())
x = np.arange(len(classes))
width = 0.25

train_vals = [train_counts[c] for c in classes]
val_vals = [val_counts.get(c, 0) for c in classes]
test_vals = [test_counts.get(c, 0) for c in classes]

bars1 = ax.bar(x - width, train_vals, width, label='Train', color='#3498db')
bars2 = ax.bar(x, val_vals, width, label='Validation', color='#2ecc71')
bars3 = ax.bar(x + width, test_vals, width, label='Test', color='#e74c3c')

ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_title('Disease Detection Dataset Distribution', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=15)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.annotate(f'{int(height)}',
                        xy=(bar.get_x() + bar.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'dataset_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved!")

## 4. Data Generators (No Data Leaking)

**Important:** Augmentation is applied ONLY to training data, not validation/test.

In [ ]:
# Training data augmentation - ONLY for train set
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Validation/Test - ONLY rescale, NO augmentation (prevents data leaking)
val_test_datagen = ImageDataGenerator(rescale=1./255)

print("Data Generators created:")
print("  Train: Augmentation enabled (rotation, flip, zoom, brightness)")
print("  Val/Test: Rescale only (NO augmentation - prevents data leaking)")

In [ ]:
# Create generators
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(CONFIG['img_size'], CONFIG['img_size']),
    batch_size=CONFIG['batch_size'],
    class_mode='categorical',
    shuffle=True
)

val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(CONFIG['img_size'], CONFIG['img_size']),
    batch_size=CONFIG['batch_size'],
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(CONFIG['img_size'], CONFIG['img_size']),
    batch_size=CONFIG['batch_size'],
    class_mode='categorical',
    shuffle=False
)

# Class mappings
class_indices = train_generator.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}

print(f"\nClass indices: {class_indices}")

## 5. Class Weights (Handle Imbalance)

In [ ]:
# Compute class weights to handle imbalance
all_classes = train_generator.classes
unique_classes = np.unique(all_classes)
weights = compute_class_weight('balanced', classes=unique_classes, y=all_classes)
class_weights = dict(zip(unique_classes, weights))

print("Class Weights (to handle imbalance):")
for idx, weight in class_weights.items():
    print(f"  {idx_to_class[idx]}: {weight:.4f}")

## 6. Build Model (Anti-Overfitting Architecture)

In [ ]:
def build_model(num_classes, img_size=224, dropout_rate=0.5, l2_reg=0.01):
    """
    Build EfficientNetB0 with anti-overfitting techniques:
    - Dropout
    - L2 Regularization
    - Batch Normalization
    """
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(img_size, img_size, 3)
    )
    
    # Freeze base model initially
    base_model.trainable = False
    
    # Build model
    inputs = keras.Input(shape=(img_size, img_size, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    
    # Dense layers with anti-overfitting
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=l2(l2_reg))(x)
    x = layers.Dropout(dropout_rate)(x)
    
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation='relu', kernel_regularizer=l2(l2_reg))(x)
    x = layers.Dropout(dropout_rate)(x)
    
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs)
    return model, base_model

# Build
model, base_model = build_model(
    num_classes=NUM_CLASSES,
    img_size=CONFIG['img_size'],
    dropout_rate=CONFIG['dropout_rate'],
    l2_reg=CONFIG['l2_reg']
)

print("Model built with anti-overfitting techniques:")
print(f"  - Dropout: {CONFIG['dropout_rate']}")
print(f"  - L2 Regularization: {CONFIG['l2_reg']}")
print(f"  - Batch Normalization: Yes")
print(f"  - Data Augmentation: Training only")

In [ ]:
# Model summary
model.summary()

In [ ]:
# Compile model
model.compile(
    optimizer=Adam(learning_rate=CONFIG['learning_rate']),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("Model compiled!")

## 7. Callbacks (Early Stopping for Overfitting Prevention)

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=CONFIG['patience_early_stop'],
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=os.path.join(MODEL_DIR, 'best_model.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=CONFIG['patience_reduce_lr'],
        min_lr=1e-7,
        verbose=1
    )
]

print("Callbacks:")
print(f"  - Early Stopping (patience={CONFIG['patience_early_stop']})")
print(f"  - Model Checkpoint (save best)")
print(f"  - Reduce LR (patience={CONFIG['patience_reduce_lr']})")

## 8. Phase 1: Train with Frozen Base

In [ ]:
print("="*60)
print("PHASE 1: Training with frozen base model")
print("="*60)

history_phase1 = model.fit(
    train_generator,
    epochs=CONFIG['epochs'] // 2,
    validation_data=val_generator,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

print("\nPhase 1 completed!")

## 9. Phase 2: Fine-tuning

In [ ]:
print("="*60)
print("PHASE 2: Fine-tuning")
print("="*60)

# Unfreeze base model
base_model.trainable = True

# Freeze first 100 layers
for layer in base_model.layers[:100]:
    layer.trainable = False

trainable_layers = sum([1 for layer in base_model.layers if layer.trainable])
print(f"Unfrozen layers: {trainable_layers}")

# Recompile with lower LR
model.compile(
    optimizer=Adam(learning_rate=CONFIG['learning_rate'] / 10),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train Phase 2
history_phase2 = model.fit(
    train_generator,
    epochs=CONFIG['epochs'] // 2,
    validation_data=val_generator,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

print("\nPhase 2 completed!")

## 10. Training Curves

In [ ]:
# Combine histories
def combine_histories(h1, h2):
    combined = {}
    for key in h1.history.keys():
        combined[key] = h1.history[key] + h2.history[key]
    return combined

history = combine_histories(history_phase1, history_phase2)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history['accuracy'], label='Train', linewidth=2)
axes[0].plot(history['val_accuracy'], label='Validation', linewidth=2)
axes[0].axvline(x=len(history_phase1.history['accuracy'])-1, color='r', linestyle='--', label='Fine-tuning Start')
axes[0].set_title('Model Accuracy', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history['loss'], label='Train', linewidth=2)
axes[1].plot(history['val_loss'], label='Validation', linewidth=2)
axes[1].axvline(x=len(history_phase1.history['loss'])-1, color='r', linestyle='--', label='Fine-tuning Start')
axes[1].set_title('Model Loss', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Training curves saved!")

In [ ]:
# Overfitting Check
final_train_acc = history['accuracy'][-1]
final_val_acc = history['val_accuracy'][-1]
gap = abs(final_train_acc - final_val_acc)

print("="*60)
print("OVERFITTING CHECK")
print("="*60)
print(f"Final Train Accuracy: {final_train_acc:.4f}")
print(f"Final Val Accuracy:   {final_val_acc:.4f}")
print(f"Gap:                  {gap:.4f}")

if gap < 0.05:
    print("\n✅ EXCELLENT: No overfitting detected!")
elif gap < 0.10:
    print("\n✅ GOOD: Minimal overfitting")
elif gap < 0.15:
    print("\n⚠️ WARNING: Some overfitting detected")
else:
    print("\n❌ ISSUE: Significant overfitting")

## 11. Model Evaluation

In [ ]:
# Load best model
best_model_path = os.path.join(MODEL_DIR, 'best_model.keras')
if os.path.exists(best_model_path):
    model = keras.models.load_model(best_model_path)
    print(f"Best model loaded from: {best_model_path}")

# Evaluate
test_generator.reset()
test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

## 12. Generate Predictions

In [ ]:
test_generator.reset()
predictions = model.predict(test_generator, verbose=1)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_generator.classes

print(f"\nPredictions: {len(predicted_classes)}")
print(f"True labels: {len(true_classes)}")

## 13. Class-wise Metrics (Supervisor Requirement)

### Requirements:
- ✅ Precision, Recall, F1-score for each class
- ✅ Values should be close to each other
- ✅ Similar values across all classes
- ✅ Accuracy close to F1

In [ ]:
# Calculate class-wise metrics
precision, recall, f1, support = precision_recall_fscore_support(
    true_classes, predicted_classes, average=None
)

# Create DataFrame
class_names_list = [idx_to_class[i] for i in range(NUM_CLASSES)]
class_metrics_df = pd.DataFrame({
    'Class': class_names_list,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support.astype(int)
})

# P-R-F1 difference
class_metrics_df['P-R-F1 Max Diff'] = class_metrics_df.apply(
    lambda row: max(row['Precision'], row['Recall'], row['F1-Score']) - 
                min(row['Precision'], row['Recall'], row['F1-Score']), axis=1
)

print("="*70)
print("CLASS-WISE METRICS")
print("="*70)
print(class_metrics_df.to_string(index=False))

# Save
class_metrics_df.to_csv(os.path.join(MODEL_DIR, 'class_metrics.csv'), index=False)
print(f"\nSaved to: class_metrics.csv")

In [ ]:
# Overall metrics
accuracy = accuracy_score(true_classes, predicted_classes)

precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    true_classes, predicted_classes, average='macro'
)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
    true_classes, predicted_classes, average='weighted'
)

print("="*70)
print("OVERALL METRICS")
print("="*70)
print(f"\nAccuracy:          {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"\n--- Macro Average ---")
print(f"Precision: {precision_macro:.4f}")
print(f"Recall:    {recall_macro:.4f}")
print(f"F1-Score:  {f1_macro:.4f}")
print(f"\n--- Weighted Average ---")
print(f"Precision: {precision_weighted:.4f}")
print(f"Recall:    {recall_weighted:.4f}")
print(f"F1-Score:  {f1_weighted:.4f}")

In [ ]:
# Supervisor Check: Accuracy vs F1
print("="*70)
print("SUPERVISOR CHECK: Accuracy vs F1-Score")
print("="*70)

acc_f1_diff = abs(accuracy - f1_macro)

print(f"\nAccuracy:         {accuracy:.4f}")
print(f"F1-Score (macro): {f1_macro:.4f}")
print(f"Difference:       {acc_f1_diff:.4f}")

if acc_f1_diff < 0.03:
    print("\n✅ EXCELLENT: Accuracy is very close to F1!")
elif acc_f1_diff < 0.05:
    print("\n✅ GOOD: Accuracy is close to F1")
elif acc_f1_diff < 0.10:
    print("\n⚠️ OK: Some difference between Accuracy and F1")
else:
    print("\n❌ ISSUE: Large gap between Accuracy and F1")

In [ ]:
# Supervisor Check: P-R-F1 Balance per Class
print("="*70)
print("SUPERVISOR CHECK: P-R-F1 Balance per Class")
print("="*70)

for i, row in class_metrics_df.iterrows():
    diff = row['P-R-F1 Max Diff']
    if diff < 0.05:
        status = "✅ EXCELLENT"
    elif diff < 0.10:
        status = "✅ GOOD"
    elif diff < 0.15:
        status = "⚠️ OK"
    else:
        status = "❌ NEEDS WORK"
    
    print(f"{status} {row['Class']}: P={row['Precision']:.3f}, R={row['Recall']:.3f}, F1={row['F1-Score']:.3f}, Diff={diff:.3f}")

In [ ]:
# Supervisor Check: Similar values across classes
print("="*70)
print("SUPERVISOR CHECK: Balance Across Classes")
print("="*70)

precision_std = np.std(class_metrics_df['Precision'])
recall_std = np.std(class_metrics_df['Recall'])
f1_std = np.std(class_metrics_df['F1-Score'])
avg_std = (precision_std + recall_std + f1_std) / 3

print(f"\nStandard Deviation Across Classes:")
print(f"  Precision: {precision_std:.4f}")
print(f"  Recall:    {recall_std:.4f}")
print(f"  F1-Score:  {f1_std:.4f}")
print(f"  Average:   {avg_std:.4f}")

if avg_std < 0.05:
    print("\n✅ EXCELLENT: Very balanced across all classes!")
elif avg_std < 0.10:
    print("\n✅ GOOD: Reasonably balanced")
elif avg_std < 0.15:
    print("\n⚠️ OK: Some imbalance")
else:
    print("\n❌ ISSUE: Significant imbalance across classes")

## 14. Classification Report

In [ ]:
print("="*70)
print("CLASSIFICATION REPORT")
print("="*70)

report = classification_report(true_classes, predicted_classes, target_names=class_names_list)
print(report)

# Save
with open(os.path.join(MODEL_DIR, 'classification_report.txt'), 'w') as f:
    f.write("CLASSIFICATION REPORT\n")
    f.write("="*70 + "\n")
    f.write(report)
    f.write(f"\nTest Accuracy: {accuracy:.4f}")

print("Saved!")

## 15. Confusion Matrix

In [ ]:
cm = confusion_matrix(true_classes, predicted_classes)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names_list, yticklabels=class_names_list, ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=class_names_list, yticklabels=class_names_list, ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved!")

## 16. Class Metrics Chart

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(NUM_CLASSES)
width = 0.25

bars1 = ax.bar(x - width, class_metrics_df['Precision'], width, label='Precision', color='#2ecc71')
bars2 = ax.bar(x, class_metrics_df['Recall'], width, label='Recall', color='#3498db')
bars3 = ax.bar(x + width, class_metrics_df['F1-Score'], width, label='F1-Score', color='#e74c3c')

def add_labels(bars):
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

add_labels(bars1)
add_labels(bars2)
add_labels(bars3)

ax.set_xlabel('Class', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Class-wise Precision, Recall, F1-Score', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(class_metrics_df['Class'], rotation=15)
ax.legend()
ax.set_ylim(0, 1.15)
ax.axhline(y=accuracy, color='purple', linestyle='--', alpha=0.7, label=f'Accuracy ({accuracy:.2f})')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'class_metrics_chart.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved!")

## 17. Radar Chart

In [ ]:
from math import pi

categories = ['Precision', 'Recall', 'F1-Score']
N = len(categories)

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

colors = plt.cm.Set2(np.linspace(0, 1, NUM_CLASSES))

for i, (_, row) in enumerate(class_metrics_df.iterrows()):
    values = [row['Precision'], row['Recall'], row['F1-Score']]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=row['Class'], color=colors[i])
    ax.fill(angles, values, alpha=0.1, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylim(0, 1)
ax.set_title('Class-wise Metrics Radar Chart', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'metrics_radar_chart.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved!")

## 18. Sample Predictions

In [ ]:
test_generator.reset()
sample_batch = next(test_generator)
sample_images, sample_labels = sample_batch

sample_preds = model.predict(sample_images, verbose=0)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('Sample Test Predictions', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < len(sample_images):
        ax.imshow(sample_images[i])
        true_idx = np.argmax(sample_labels[i])
        pred_idx = np.argmax(sample_preds[i])
        conf = sample_preds[i][pred_idx] * 100
        
        color = 'green' if true_idx == pred_idx else 'red'
        ax.set_title(f'True: {idx_to_class[true_idx]}\nPred: {idx_to_class[pred_idx]} ({conf:.1f}%)',
                     fontsize=9, color=color)
        ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'sample_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved!")

## 19. Save Model Info

In [ ]:
model_info = {
    'model_name': 'Coconut Leaf Disease Detection',
    'version': MODEL_VERSION,
    'created_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'base_model': 'EfficientNetB0',
    'input_shape': [CONFIG['img_size'], CONFIG['img_size'], 3],
    'num_classes': NUM_CLASSES,
    'classes': class_names_list,
    'class_indices': class_indices,
    'training_config': CONFIG,
    'performance': {
        'test_accuracy': float(accuracy),
        'test_loss': float(test_loss),
        'precision_macro': float(precision_macro),
        'recall_macro': float(recall_macro),
        'f1_macro': float(f1_macro),
        'precision_weighted': float(precision_weighted),
        'recall_weighted': float(recall_weighted),
        'f1_weighted': float(f1_weighted),
        'accuracy_f1_diff': float(acc_f1_diff)
    },
    'class_metrics': {
        class_name: {
            'precision': float(class_metrics_df[class_metrics_df['Class']==class_name]['Precision'].values[0]),
            'recall': float(class_metrics_df[class_metrics_df['Class']==class_name]['Recall'].values[0]),
            'f1_score': float(class_metrics_df[class_metrics_df['Class']==class_name]['F1-Score'].values[0]),
            'support': int(class_metrics_df[class_metrics_df['Class']==class_name]['Support'].values[0])
        }
        for class_name in class_names_list
    },
    'supervisor_checks': {
        'no_data_leaking': True,
        'overfitting_gap': float(gap),
        'accuracy_f1_close': acc_f1_diff < 0.05,
        'class_balance_std': float(avg_std)
    }
}

with open(os.path.join(MODEL_DIR, 'model_info.json'), 'w') as f:
    json.dump(model_info, f, indent=2)

print("Model info saved!")
print(json.dumps(model_info, indent=2))

## 20. Final Summary

In [ ]:
print("="*70)
print("FINAL MODEL TRAINING SUMMARY")
print("="*70)
print(f"\nModel: {MODEL_NAME}_{MODEL_VERSION}")
print(f"Classes: {class_names_list}")

print(f"\n--- Performance ---")
print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"F1-Score (macro): {f1_macro:.4f}")
print(f"F1-Score (weighted): {f1_weighted:.4f}")

print(f"\n--- Supervisor Requirements ---")
print(f"✅ No Data Leaking: Augmentation on train only")
print(f"{'✅' if gap < 0.1 else '⚠️'} No Overfitting: Gap = {gap:.4f}")
print(f"{'✅' if acc_f1_diff < 0.05 else '⚠️'} Accuracy ≈ F1: Diff = {acc_f1_diff:.4f}")
print(f"{'✅' if avg_std < 0.1 else '⚠️'} Balanced Classes: Std = {avg_std:.4f}")

print(f"\n--- Class-wise Metrics ---")
print(class_metrics_df.to_string(index=False))

print(f"\n--- Saved Files ---")
print(f"Model: {MODEL_DIR}/best_model.keras")
print(f"Info: {MODEL_DIR}/model_info.json")
print(f"Metrics: {MODEL_DIR}/class_metrics.csv")
print(f"Charts: dataset_distribution.png, training_curves.png, confusion_matrix.png, etc.")

print("\n" + "="*70)
print("TRAINING COMPLETED!")
print("="*70)